In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline

SEED = 42
warnings.filterwarnings("ignore", category=UserWarning)  # MLP convergence noise

In [ ]:
def load_uci():
    """UCI Online Shoppers Purchasing Intention (12,330 sessions, target = Revenue)."""
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=468)
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    return prepare_uci(df)


def prepare_uci(df):
    df = df.copy()
    df["Weekend"] = df["Weekend"].astype(int)
    y = df.pop("Revenue").astype(int)
    X = pd.get_dummies(df, columns=["Month", "VisitorType"], dtype=int)
    return X, y


def load_cosmetics(csv_glob="~/.cache/kagglehub/**/20*.csv", n_rows=12_000):
    """
    Kaggle 'eCommerce Events History in Cosmetics Shop' (Kechinov, 2020).
    Downloads the monthly CSVs (2019-Oct ... 2020-Feb), randomly samples
    12,000 event rows, then rebuilds session-level features.
    """
    import kagglehub
    path = kagglehub.dataset_download("mkechinov/ecommerce-events-history-in-cosmetics-shop")
    files = sorted(glob.glob(f"{path}/*.csv"))
    per_file = n_rows // len(files)
    events = pd.concat(
        [pd.read_csv(f).sample(n=per_file, random_state=SEED) for f in files],
        ignore_index=True,
    )
    return prepare_cosmetics(events)


def prepare_cosmetics(events):
    """Event log -> one row per user_session."""
    ev = events.copy()
    ev["event_time"] = pd.to_datetime(ev["event_time"], utc=True)
    g = ev.groupby("user_session")

    sess = pd.DataFrame({
        "ViewCount": g["event_type"].apply(lambda s: (s == "view").sum()),
        "CartCount": g["event_type"].apply(lambda s: (s == "cart").sum()),
        "RemoveFromCartCount": g["event_type"].apply(lambda s: (s == "remove_from_cart").sum()),
        "TotalEvents": g.size(),
        "SessionDurationSeconds": g["event_time"].apply(lambda t: (t.max() - t.min()).total_seconds()),
        "DistinctProducts": g["product_id"].nunique(),
        "DistinctCategories": g["category_id"].nunique(),
        "DistinctBrands": g["brand"].nunique(),
        "AvgPrice": g["price"].mean(),
        "MaxPrice": g["price"].max(),
        "Purchase": g["event_type"].apply(lambda s: int((s == "purchase").any())),
    })
    sess["CartAbandonRate"] = (
        sess["RemoveFromCartCount"] / sess["CartCount"].replace(0, np.nan)
    ).fillna(0)

    # NOTE: if purchase events are included in TotalEvents the label leaks into a
    # feature. Recompute it excluding purchases:
    non_purchase = ev[ev["event_type"] != "purchase"].groupby("user_session").size()
    sess["TotalEvents"] = non_purchase.reindex(sess.index).fillna(0)

    y = sess.pop("Purchase")
    return sess, y

In [ ]:
def build_models(balanced=False):
    """
    Weka ClassBalancer re-weights instances so each class has equal total weight.
    sklearn equivalent: class_weight='balanced' (LR, RF). MLPClassifier has no
    class_weight, so oversample the minority class inside each CV training fold.
    """
    cw = "balanced" if balanced else None
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight=cw))
    rf = RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=SEED, n_jobs=-1)
    mlp_core = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=SEED)
    if balanced:
        mlp = make_imb_pipeline(StandardScaler(), RandomOverSampler(random_state=SEED), mlp_core)
    else:
        mlp = make_pipeline(StandardScaler(), mlp_core)
    return {"Logistic Regression": lr, "Random Forest": rf, "Multilayer Perceptron": mlp}


def evaluate(X, y, balanced=False, label=""):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    rows = []
    for name, model in build_models(balanced).items():
        proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
        pred = (proba >= 0.5).astype(int)
        rows.append({
            "Model": f"{name}{' + balancing' if balanced else ''}",
            "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),   # minority class
            "Recall": recall_score(y, pred),
            "F1": f1_score(y, pred),
            "ROC-AUC": roc_auc_score(y, proba),
        })
    out = pd.DataFrame(rows).round(3)
    print(f"\n=== Classification {label} ===\n{out.to_string(index=False)}")
    return out


In [ ]:
def classes_to_clusters_error(X_scaled, y, k):
    """Weka 'classes to clusters evaluation': map each cluster to its majority class,
    report % of instances in the 'wrong' cluster."""
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_scaled)
    ct = pd.crosstab(km.labels_, y)
    wrong = ct.sum().sum() - ct.max(axis=1).sum()
    return 100 * wrong / len(y), km.labels_


def clustering_analysis(X, y, label=""):
    Xs = StandardScaler().fit_transform(X)
    results, labels_k2 = [], None
    for k in range(2, 6):
        err, labels = classes_to_clusters_error(Xs, y, k)
        results.append({"k": k, "% incorrectly clustered": round(err, 2)})
        if k == 2:
            labels_k2 = labels
    print(f"\n=== K-means {label} ===\n{pd.DataFrame(results).to_string(index=False)}")

    # Add k=2 cluster label as an extra feature and re-run the classifiers
    X_c = X.copy()
    X_c["Cluster"] = labels_k2
    return X_c

In [ ]:
def feature_ranking(X, y, top=5, label=""):
    """
    Weka InfoGainAttributeEval ~ mutual information between attribute and class.
    (Weka discretises numerics with MDL; sklearn estimates MI with k-NN, so values
    differ in scale but the ranking is usually similar.)
    """
    mi = pd.Series(mutual_info_classif(X, y, random_state=SEED), index=X.columns)
    rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X, y)
    rf_imp = pd.Series(rf.feature_importances_, index=X.columns)
    out = pd.DataFrame({"InfoGain (MI)": mi, "RF importance": rf_imp}).sort_values(
        "InfoGain (MI)", ascending=False)
    print(f"\n=== Top {top} predictors {label} ===\n{out.head(top).round(4)}")
    return out


In [ ]:
def run_pipeline(X, y, name):
    print(f"\n######## {name}: {len(y)} rows, positive rate = {y.mean():.2%} ########")
    evaluate(X, y, balanced=False, label=f"{name} (baseline)")
    evaluate(X, y, balanced=True, label=f"{name} (class-balanced)")
    X_c = clustering_analysis(X, y, label=name)
    evaluate(X_c, y, balanced=True, label=f"{name} (+ cluster feature, balanced)")
    feature_ranking(X, y, label=name)


if __name__ == "__main__":
    X_uci, y_uci = load_uci()
    run_pipeline(X_uci, y_uci, "UCI Online Shoppers")

    X_cos, y_cos = load_cosmetics()
    run_pipeline(X_cos, y_cos, "Cosmetics Shop")
